In [1]:
import rioxarray
from shapely.geometry import box
import pandas as pd
import shutil
import os
import requests
from tqdm import tqdm
from zipfile import ZipFile
from rasterio.enums import Resampling
from datetime import datetime
import numpy as np

In [ ]:
big = pd.read_csv('./Camcore_Pine_dataset.csv')
big["date_range"] = big["data_final"].astype(str) + ", " + big["date_final"].astype(str)
required_view = big.groupby(['latitude', 'longitude'])['date_range'].unique().reset_index()
required_view.head()

,latitude,longitude,date_range
0,-38.600000,-72.400000,"[9/1/1996, 8/30/2004]"
1,-37.409167,-73.266944,"[7/6/1995, 7/4/2003]"
2,-37.033333,-72.233333,"[7/1/1997, 6/29/2005]"
3,-35.870833,-71.975278,"[8/1/1997, 7/30/2005]"
4,-35.306944,-72.394722,"[7/1/1993, 6/29/2001]"


In [79]:
required_view['date_range'].apply(lambda x: x[0].split(",")[1].strip())

0      8/30/2004
1       7/4/2003
2      6/29/2005
3      7/30/2005
4      6/29/2001
         ...    
305    5/30/2002
306    6/29/1992
307    6/29/1993
308    9/29/1993
309     7/9/2011
Name: date_range, Length: 310, dtype: object

In [80]:
required_view['data_final'] = required_view['date_range'].apply(lambda x: x[0].split(",")[0].strip())
required_view['date_final'] = required_view['date_range'].apply(lambda x: x[0].split(",")[1].strip())

In [81]:
required_view[["latitude", "longitude","data_final","date_final"]].drop_duplicates().value_counts()

latitude    longitude   data_final  date_final
 17.814889  -93.754444  7/11/2003   7/9/2011      1
-38.600000  -72.400000  9/1/1996    8/30/2004     1
-37.409167  -73.266944  7/6/1995    7/4/2003      1
-37.033333  -72.233333  7/1/1997    6/29/2005     1
-35.870833  -71.975278  8/1/1997    7/30/2005     1
                                                 ..
-34.020750   23.080900  6/25/2014   6/23/2022     1
-34.030386   23.181044  6/11/2013   6/9/2021      1
-34.030736   24.172700  10/9/2014   10/7/2022     1
-34.038028   23.169333  12/2/2008   11/30/2016    1
-34.040703   23.174753  4/30/2009   4/28/2017     1
Name: count, Length: 310, dtype: int64

In [3]:
def get_season(month):
    month = int(month)
    season = ""
    if month in [12, 1, 2]:
        season =  "Summer"
    elif month in [3, 4, 5]:
        season = "Autumn"
    elif month in [6, 7, 8]:
        season = "Winter"
    else:
        season = "Spring"
    return season

In [13]:
def process_tifs(data_types, root_dir, all_data, start_date, end_date, lat_center, lon_center, lat_size=0.05, lon_size=0.05):
    for data_type in data_types:
        data_type_path = os.path.join(root_dir, data_type)
        
        for year_block in os.listdir(data_type_path):
            year_block_path = os.path.join(data_type_path, year_block)

            for year in os.listdir(year_block_path):
                try:
                    year_int = int(year)
                    if year_int < start_date.year or year_int > end_date.year:
                        continue  # Skip years outside the range
                except ValueError:
                    continue  # Skip if the folder name is not a valid year
                
                year_path = os.path.join(year_block_path, year)

                # Determine valid months for the given year
                if year_int == start_date.year:
                    start_month = start_date.month
                else:
                    start_month = 1  # Default to January

                if year_int == end_date.year:
                    end_month = end_date.month
                else:
                    end_month = 12  # Default to December

                for month in range(start_month, end_month + 1):  # Process only valid months
                    month_str = f"{month:02d}"
                    date_str = f"{year}-{month_str}-01"  # Construct YYYY-MM-DD format

                    tif_filename = f"wc2.1_2.5m_{data_type}_{year}-{month_str}.tif"
                    tif_filepath = os.path.join(year_path, tif_filename)

                    if os.path.exists(tif_filepath):
                        # Get the bounding box coordinates
                        lat_min = lat_center - (lat_size / 2)
                        lat_max = lat_center + (lat_size / 2)
                        lon_min = lon_center - (lon_size / 2)
                        lon_max = lon_center + (lon_size / 2)
                        try:
                            xds__2p5m = rioxarray.open_rasterio(
                                tif_filepath, mask_and_scale=True
                            )
                            xds__2p5m_clipped = xds__2p5m.rio.clip_box(
                                minx=lon_min,
                                miny=lat_min,
                                maxx=lon_max,
                                maxy=lat_max,
                                allow_one_dimensional_raster=True
                            )
                            xds_match_30s_sample = rioxarray.open_rasterio("./wc2.1_30s_prec_01.tif",mask_and_scale=True,)
                            xds_match_30s_clipped_sample = xds_match_30s_sample.rio.clip_box(
                            minx=lon_min,
                            miny=lat_min,
                            maxx=lon_max,
                            maxy=lat_max,
                            allow_one_dimensional_raster=True,
                            )
                            xds__2p5m_clipped_upsampled = xds__2p5m_clipped.rio.reproject_match(
                                xds_match_30s_clipped_sample, resampling=Resampling.cubic_spline
                            )
                            rds = xds__2p5m_clipped_upsampled.squeeze().drop_vars("spatial_ref").drop_vars("band")
                            rds.name = f"{data_type}_{year}_{month_str}"
                            df = rds.to_dataframe().reset_index()
                            df[f'season_{rds.name}'] = get_season(month)
                            all_data[f"{data_type}"].append(df)
                        except Exception as e:
                            print(f"Error processing {tif_filename}: {e}")
                            continue
    return all_data

In [14]:
def get_closest_row(all_data, lat_center, lon_center):
    all_data["prec"][0]["distance"] = np.sqrt((all_data["prec"][0]["y"] - lat_center) ** 2 + (all_data["prec"][0]["x"] - lon_center) ** 2)
    all_data["prec"][0].head()  
    # Select the row with the minimum distance
    closest_row = all_data["prec"][0].loc[all_data["prec"][0]["distance"].idxmin()]

    all_data["prec"][0].drop(columns=["distance"], inplace=True)
    return closest_row

In [15]:
def filter_and_merge_data(closest_row, all_data):
    filtered_dfs_prec = [df[(df["x"] == closest_row["x"]) & (df["y"] == closest_row["y"])] for df in all_data["prec"]]
    filtered_dfs_tmax = [df[(df["x"] == closest_row["x"]) & (df["y"] == closest_row["y"])] for df in all_data["tmax"]]
    filtered_dfs_tmin = [df[(df["x"] == closest_row["x"]) & (df["y"] == closest_row["y"])] for df in all_data["tmin"]]
    merged_df_prec = filtered_dfs_prec[0]  # Start with the first dataframe

    # Merge all the dataframes
    for df in filtered_dfs_prec[1:]:
        merged_df_prec = pd.merge(merged_df_prec, df, on=["x", "y"], how="outer")
    merged_df_tmax = filtered_dfs_tmax[0]  # Start with the first dataframe

    # Merge all the dataframes
    for df in filtered_dfs_tmax[1:]:
        merged_df_tmax = pd.merge(merged_df_tmax, df, on=["x", "y"], how="outer")
    
    merged_df_tmin = filtered_dfs_tmin[0]  # Start with the first dataframe

    # Merge all the dataframes
    for df in filtered_dfs_tmin[1:]:
        merged_df_tmin = pd.merge(merged_df_tmin, df, on=["x", "y"], how="outer")
    return merged_df_prec, merged_df_tmax, merged_df_tmin

In [16]:
def merge_dfs(merged_df_prec, merged_df_tmax, merged_df_tmin):
    id_vars = ['y', 'x']  # Columns that should remain unchanged
    value_vars = [col for col in merged_df_prec.columns if col not in id_vars]  # All precipitation and season columns

    # Melting the DataFrame
    df_long = pd.melt(merged_df_prec, id_vars=id_vars, value_vars=value_vars, var_name="variable", value_name="value")

    # Extracting year, month, and type (prec or season) from column names
    df_long[['type', 'year', 'month']] = df_long['variable'].str.extract(r'(\w+)_(\d{4})_(\d{2})')

    # Pivoting to get separate 'prec' and 'season' columns
    prec_df = df_long.pivot_table(index=['y', 'x', 'year', 'month'], columns='type', values='value', aggfunc='first').reset_index()

    # Renaming columns
    prec_df.columns.name = None
    prec_df = prec_df.rename(columns={'prec': 'precipitation', 'season': 'season'})

    # Convert year and month to integers
    prec_df['year'] = prec_df['year'].astype(int)
    prec_df['month'] = prec_df['month'].astype(int)
    value_vars = [col for col in merged_df_tmax.columns if col not in id_vars]  # All precipitation and season columns

    # Melting the DataFrame
    df_long = pd.melt(merged_df_tmax, id_vars=id_vars, value_vars=value_vars, var_name="variable", value_name="value")

    # Extracting year, month, and type (prec or season) from column names
    df_long[['type', 'year', 'month']] = df_long['variable'].str.extract(r'(\w+)_(\d{4})_(\d{2})')

    # Pivoting to get separate 'prec' and 'season' columns
    tmax_df = df_long.pivot_table(index=['y', 'x', 'year', 'month'], columns='type', values='value', aggfunc='first').reset_index()

    # Renaming columns
    tmax_df.columns.name = None
    tmax_df = tmax_df.rename(columns={'tmax': 'maxtemperature', 'season': 'season'})

    # Convert year and month to integers
    tmax_df['year'] = tmax_df['year'].astype(int)
    tmax_df['month'] = tmax_df['month'].astype(int)

    value_vars = [col for col in merged_df_tmin.columns if col not in id_vars]  # All precipitation and season columns

    # Melting the DataFrame
    df_long = pd.melt(merged_df_tmin, id_vars=id_vars, value_vars=value_vars, var_name="variable", value_name="value")

    # Extracting year, month, and type (prec or season) from column names
    df_long[['type', 'year', 'month']] = df_long['variable'].str.extract(r'(\w+)_(\d{4})_(\d{2})')

    # Pivoting to get separate 'prec' and 'season' columns
    tmin_df = df_long.pivot_table(index=['y', 'x', 'year', 'month'], columns='type', values='value', aggfunc='first').reset_index()

    # Renaming columns
    tmin_df.columns.name = None
    tmin_df = tmin_df.rename(columns={'tmin': 'mintemperature', 'season': 'season'})

    # Convert year and month to integers
    tmin_df['year'] = tmin_df['year'].astype(int)
    tmin_df['month'] = tmin_df['month'].astype(int)

    # Merge the DataFrames
    merged_df = prec_df.merge(tmax_df, on=['y', 'x', 'year', 'month'], how='inner') \
                   .merge(tmin_df, on=['y', 'x', 'year', 'month'], how='inner')
    merged_df.drop(columns=["x","y","season_tmax","season_tmin"], inplace=True,axis=1)
    merged_df.rename(columns={"season_prec":"season"}, inplace=True)
    return merged_df





In [17]:
def get_covariables(merged_df, lat_center, lon_center, date_range):
    res = merged_df.groupby(["year","season"])['precipitation'].max().groupby("season").mean()
    wm_prec1_autumn = res.iloc[0]  # Autumn
    wm_prec1_spring = res.iloc[1]  # Spring
    wm_prec1_summer = res.iloc[2]  # Summer
    wm_prec1_winter = res.iloc[3]  # Winter
    res = merged_df.groupby(["year","season"])['precipitation'].median().groupby("season").mean()
    wm_prec2_autumn = res.iloc[0]  # Autumn
    wm_prec2_spring = res.iloc[1]  # Spring
    wm_prec2_summer = res.iloc[2]  # Summer
    wm_prec2_winter = res.iloc[3]  # Winter
    res = merged_df.groupby(["year","season"])['maxtemperature'].max().groupby("season").mean()
    wm_tmax1_autumn = res.iloc[0]  # Autumn
    wm_tmax1_spring = res.iloc[1]  # Spring
    wm_tmax1_summer = res.iloc[2]  # Summer
    wm_tmax1_winter = res.iloc[3]  # Winter
    res = merged_df.groupby(["year","season"])['mintemperature'].min().groupby("season").mean()
    wm_tmin1_autumn = res.iloc[0]  # Autumn
    wm_tmin1_spring = res.iloc[1]  # Spring
    wm_tmin1_summer = res.iloc[2]  # Summer
    wm_tmin1_winter = res.iloc[3]  # Winter
    res = merged_df.groupby(["year","season"])['maxtemperature'].max().groupby("season").min()
    wm_tmax2_autumn = res.iloc[0]  # Autumn
    wm_tmax2_spring = res.iloc[1]  # Spring
    wm_tmax2_summer = res.iloc[2]  # Summer
    wm_tmax2_winter = res.iloc[3]  # Winter
    merged_df['diurnal_range'] = merged_df['maxtemperature'] - merged_df['mintemperature']

    mean_diurnal_range = merged_df.groupby(['year', 'month'])['diurnal_range'].mean().groupby('month').mean().mean()

    wm_bio2 = mean_diurnal_range  
    wm_bio5 = merged_df.groupby(["year"])["maxtemperature"].max().mean()
    wm_bio6 = merged_df.groupby(["year"])["mintemperature"].min().mean()
    wm_bio7 = (merged_df.groupby(["year"])["maxtemperature"].max() -  merged_df.groupby(["year"])["mintemperature"].min()).mean()
    # wm_bio3 = merged_df.groupby(['year', 'month'])['diurnal_range'].mean().groupby('month').mean() 
    a = list(merged_df.groupby(['year', 'month'])['diurnal_range'].mean().groupby('month').mean()) 
    b = list(merged_df.groupby(["year"])["maxtemperature"].max() -  merged_df.groupby(["year"])["mintemperature"].min())
    wm_bio3 = np.mean([a[i] / b[i] for i in range(len(a) if len(a) < len(b) else len(b))])
    wm_bio12 = merged_df.groupby(["year"])["precipitation"].sum().mean()
    wm_bio13 = merged_df.groupby(["year"])["precipitation"].max().mean()
    wm_bio14 = merged_df.groupby(["year"])["precipitation"].min().mean()
    merged_df['quarter'] = (merged_df['month'] - 1) // 3 + 1
    temp_df = merged_df.groupby(['year', 'quarter'])['precipitation'].sum()
    wm_bio16 = temp_df[temp_df.groupby("year").idxmax()].mean()
    merged_df['quarter'] = (merged_df['month'] - 1) // 3 + 1
    temp_df = merged_df.groupby(['year', 'quarter'])['precipitation'].sum()
    wm_bio17 = temp_df[temp_df.groupby("year").idxmin()].mean()
    temp_df_tmax = merged_df.groupby(['year', 'quarter'])['maxtemperature'].sum()
    temp_df_prec = merged_df.groupby(['year', 'quarter'])['precipitation'].sum()
    wm_bio18 = temp_df_prec[temp_df_tmax.groupby("year").idxmin()].mean()
    temp_df_tmax = merged_df.groupby(['year', 'quarter'])['mintemperature'].sum()
    temp_df_prec = merged_df.groupby(['year', 'quarter'])['precipitation'].sum()
    wm_bio19 = temp_df_prec[temp_df_tmax.groupby("year").idxmin()].mean()
    data = {
    "latitude": lat_center, 
    "longitude": lon_center,
    'wm_prec1_autumn': wm_prec1_autumn,
    'wm_prec1_spring': wm_prec1_spring,
    'wm_prec1_summer': wm_prec1_summer,
    'wm_prec1_winter': wm_prec1_winter,
    'wm_prec2_autumn': wm_prec2_autumn,
    'wm_prec2_spring': wm_prec2_spring,
    'wm_prec2_summer': wm_prec2_summer,
    'wm_prec2_winter': wm_prec2_winter,
    'wm_tmax1_autumn': wm_tmax1_autumn,
    'wm_tmax1_spring': wm_tmax1_spring,
    'wm_tmax1_summer': wm_tmax1_summer,
    'wm_tmax1_winter': wm_tmax1_winter,
    'wm_tmin1_autumn': wm_tmin1_autumn,
    'wm_tmin1_spring': wm_tmin1_spring,
    'wm_tmin1_summer': wm_tmin1_summer,
    'wm_tmin1_winter': wm_tmin1_winter,
    'wm_tmax2_autumn': wm_tmax2_autumn,
    'wm_tmax2_spring': wm_tmax2_spring,
    'wm_tmax2_summer': wm_tmax2_summer,
    'wm_tmax2_winter': wm_tmax2_winter,
    'wm_bio2': wm_bio2,
    'wm_bio5': wm_bio5,
    'wm_bio6': wm_bio6,
    'wm_bio7': wm_bio7,
    'wm_bio3': wm_bio3,
    'wm_bio12': wm_bio12,
    'wm_bio13': wm_bio13,
    'wm_bio14': wm_bio14,
    'wm_bio16': wm_bio16,
    'wm_bio17': wm_bio17,
    'wm_bio18': wm_bio18,
    'wm_bio19': wm_bio19, 
    "data_final" :  date_range[0],
    "date_final" :  date_range[1]       
    }

    df = pd.DataFrame([data])
    return df


In [18]:
data_types = ["prec", "tmax", "tmin"]
root_dir = "./WorldClimSubset/"
all_data = {"prec": [], "tmax": [], "tmin": []}
lat_center, lon_center = -38.600000, -72.400000	

date_range = ["9/1/1996", "8/30/2004"]

# Check the result
date_format = "%m/%d/%Y"  # Format used in the date strings
start_date = datetime.strptime(date_range[0], date_format).date()
end_date = datetime.strptime(date_range[1], date_format).date()


all_data = process_tifs(data_types, root_dir, all_data,start_date, end_date, lat_center, lon_center)
closest_row = get_closest_row(all_data, lat_center, lon_center)

merged_df_prec, merged_df_tmax, merged_df_tmin = filter_and_merge_data(closest_row, all_data)

merged_df = merge_dfs(merged_df_prec, merged_df_tmax, merged_df_tmin)

merged_df.head()

covars_df = get_covariables(merged_df, lat_center, lon_center, date_range)
covars_df.head()

,latitude,longitude,wm_prec1_autumn,wm_prec1_spring,wm_prec1_summer,wm_prec1_winter,wm_prec2_autumn,wm_prec2_spring,wm_prec2_summer,wm_prec2_winter,...,wm_bio3,wm_bio12,wm_bio13,wm_bio14,wm_bio16,wm_bio17,wm_bio18,wm_bio19,data_final,date_final
0,-38.6,-72.4,159.771858,153.966496,60.852583,326.154303,79.575621,81.201653,29.641108,214.564616,...,0.530812,1208.57347,301.741827,13.282245,509.672407,100.791624,439.426175,441.487279,9/1/1996,8/30/2004


In [22]:
def download_tiff(url, lat_min, lat_max, lon_min, lon_max, download_folder="./SoilGrids"):

    # Format the URL with the coordinates
    formatted_url = url.format(lat_min=lat_min, lat_max=lat_max, lon_min=lon_min, lon_max=lon_max)

    # Send a GET request to the URL
    response = requests.get(formatted_url)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Extract the file name from the URL (assumes a coverage ID is present)
        try:
            coverage_id = formatted_url.split("COVERAGEID=")[1].split("&")[0]
        except IndexError:
            coverage_id = "unknown_coverage"

        file_name = f"{coverage_id}.tif"
        # Ensure the download folder exists
        os.makedirs(download_folder, exist_ok=True)


        # Full path to save the file
        file_path = os.path.join(download_folder, file_name)

        # Save the content (image) to a file
        with open(file_path, "wb") as f:
            f.write(response.content)

        # print(f"Downloaded: {file_path}")
    else:
        print(f"Failed to download {formatted_url}. Status code: {response.status_code}")


In [20]:
def get_soil_grids_cols(lat_center, lon_center, lat_size=0.05, lon_size=0.05):
    urls = [
        "https://maps.isric.org/mapserv?map=/map/ocd.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=ocd_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/ocs.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=ocs_0-30cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/bdod.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=bdod_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/clay.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=clay_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/cfvo.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=cfvo_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/sand.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=sand_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/silt.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=silt_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/wv0010.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=wv0010_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/wv0033.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=wv0033_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/wv1500.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=wv1500_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/cec.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=cec_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/nitrogen.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=nitrogen_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/soc.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=soc_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/phh2o.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=phh2o_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
        "https://maps.isric.org/mapserv?map=/map/wrb.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=MostProbable&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    ]
    # Loop through each URL and download the corresponding TIFF file
    for url in urls:
        lat_min = lat_center - (lat_size / 2)
        lat_max = lat_center + (lat_size / 2)
        lon_min = lon_center - (lon_size / 2)
        lon_max = lon_center + (lon_size / 2)
        download_tiff(url, lat_min, lat_max, lon_min, lon_max)

    final_df = None

    # Iterate through the directory to read all the .tif files
    for root, dirs, files in os.walk("./SoilGrids/"):
        for file in files:
            if file.endswith(".tif") and file!= "MostProbable.tif":
                # Extract the variable name from the file name
                name_var = file.split("_")[0]

                # Open the raster file using rioxarray
                raster = rioxarray.open_rasterio(
                    f"./SoilGrids/{file}",
                    mask_and_scale=True,
                )

                # Squeeze to remove singleton dimensions and drop unnecessary variables
                rds = raster.squeeze().drop_vars("spatial_ref").drop_vars("band")
                rds.name = name_var

                # Convert the raster to a DataFrame
                df = rds.to_dataframe().reset_index()

                # Filter out rows with NaN values in the raster data column
                df_filtered = df[df[rds.name].notna()]

                # # Create a 'location' column using latitude and longitude
                # df_filtered['location'] = df_filtered.apply(
                #     lambda row: f"lat{row['x']:.5f}_lon{row['y']:.5f}", axis=1
                # )

                # Drop the 'x' and 'y' columns as they are not needed anymore
                # df_filtered.drop(columns=["x", "y"], inplace=True, axis=1)

                # Select only the 'location' and the current variable column
                df = df_filtered[['x','y', rds.name]]

                if final_df is None:
                    final_df = df
                else:
                    final_df = final_df.merge(df, on=['x','y'], how='outer')
    final_df.rename(columns={"x":"longitude","y":"latitude"}, inplace=True)
    return final_df

In [21]:
soil = get_soil_grids_cols(lat_center, lon_center)
soil_filtered = soil[(soil["longitude"].round(2) == closest_row["x"].round(2)) & (soil["latitude"].round(2) == closest_row["y"].round(2))]
soil_filtered["longitude"] = closest_row["x"]
soil_filtered["latitude"] = closest_row["y"]
non_zero_row = soil_filtered[(soil_filtered.iloc[:, 3:] != 0).any(axis=1)].iloc[0]
non_zero_row

Downloaded: ./SoilGrids\ocd_0-5cm_mean.tif
Downloaded: ./SoilGrids\ocs_0-30cm_mean.tif
Downloaded: ./SoilGrids\bdod_0-5cm_mean.tif
Downloaded: ./SoilGrids\clay_0-5cm_mean.tif
Downloaded: ./SoilGrids\cfvo_0-5cm_mean.tif
Downloaded: ./SoilGrids\sand_0-5cm_mean.tif
Downloaded: ./SoilGrids\silt_0-5cm_mean.tif
Downloaded: ./SoilGrids\wv0010_0-5cm_mean.tif
Downloaded: ./SoilGrids\wv0033_0-5cm_mean.tif
Downloaded: ./SoilGrids\wv1500_0-5cm_mean.tif
Downloaded: ./SoilGrids\cec_0-5cm_mean.tif
Downloaded: ./SoilGrids\nitrogen_0-5cm_mean.tif
Downloaded: ./SoilGrids\soc_0-5cm_mean.tif
Downloaded: ./SoilGrids\phh2o_0-5cm_mean.tif
Downloaded: ./SoilGrids\MostProbable.tif


C:\Users\harsh\AppData\Local\Temp\ipykernel_38868\4236135649.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  soil_filtered["longitude"] = closest_row["x"]
C:\Users\harsh\AppData\Local\Temp\ipykernel_38868\4236135649.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  soil_filtered["latitude"] = closest_row["y"]


longitude    -72.404167
latitude     -38.604167
bdod         107.000000
cec          366.000000
cfvo          90.000000
clay         277.000000
nitrogen     616.000000
ocd          737.000000
ocs          133.000000
phh2o         58.000000
sand         194.000000
silt         529.000000
soc          980.000000
wv0010       428.000000
wv0033       392.000000
wv1500       211.000000
Name: 197, dtype: float64

In [84]:
# Initialize a list to store results
all_covars = []
count_no_rows = 0
# Iterate through each row in required_view
for index, row in required_view.iterrows():
    lat_center, lon_center = row["latitude"], row["longitude"]
    lat_min = lat_center - 0.05 / 2
    lat_max = lat_center + 0.05 / 2
    lon_min = lon_center - 0.05 / 2
    lon_max = lon_center + 0.05 / 2
    date_range = row["date_range"]
    # Split the single string in the ndarray into two separate dates
    date_range_split = date_range[0].split(",")  # Split by comma
    date_range_split = [d.strip() for d in date_range_split]  # Remove leading/trailing spaces
    print(f"Processing: {lat_center}, {lon_center}, {date_range_split[0]} to {date_range_split[1]}")

    # Convert date_range to start_date and end_date
    date_format = "%m/%d/%Y"
    start_date = datetime.strptime(date_range_split[0], date_format).date()
    end_date = datetime.strptime(date_range_split[1], date_format).date()

    # Initialize data storage
    all_data = {"prec": [], "tmax": [], "tmin": []}

    # Process TIFs for the current location and date range
    all_data = process_tifs(data_types, root_dir, all_data, start_date, end_date, lat_center, lon_center)

    # Find the closest row in the dataset
    closest_row = get_closest_row(all_data, lat_center, lon_center)

    # Filter and merge data
    merged_df_prec, merged_df_tmax, merged_df_tmin = filter_and_merge_data(closest_row, all_data)

    merged_df = merge_dfs(merged_df_prec, merged_df_tmax, merged_df_tmin)
    # Get covariables and store the result
    covars_df = get_covariables(merged_df, lat_center, lon_center, date_range_split)

    # Convert the result into a dictionary (assuming it's a single-row DataFrame)
    covars_dict = covars_df.iloc[0].to_dict()  # Convert first row to dict

    # Append the computed covariates as new columns to required_view
    for col, val in covars_dict.items():
        required_view.loc[index, col] = val  # Assign each covariate
     # Get the soil data for the current location
    soil = get_soil_grids_cols(lat_center, lon_center)
    soil_filtered = soil[(soil["longitude"].round(2) == closest_row["x"].round(2)) & 
                         (soil["latitude"].round(2) == closest_row["y"].round(2))]

    # Update longitude and latitude in filtered soil data
    # soil_filtered["longitude"] = closest_row["x"]
    # soil_filtered["latitude"] = closest_row["y"]
    soil_filtered.loc[:, "longitude"] = closest_row["x"]
    soil_filtered.loc[:, "latitude"] = closest_row["y"]

    # # Extract the first non-zero row
    # non_zero_row = soil_filtered[(soil_filtered.iloc[:, 3:] != 0).any(axis=1)].iloc[0]
    # Filter non-zero rows
    non_zero_rows = soil_filtered[(soil_filtered.iloc[:, 3:] != 0).any(axis=1)]

    # Check if the resulting DataFrame is empty before indexing
    if not non_zero_rows.empty:
        non_zero_row = non_zero_rows.iloc[0]
        # Append the non-zero row variables as new columns to required_view
        for col, val in non_zero_row.items():
            required_view[col] = val
    else:
        count_no_rows += 1
        print("No non-zero rows found in soil_filtered.")

    # Append the non-zero row variables as new columns to required_view
    for col, val in non_zero_row.items():
        if col not in ["longitude", "latitude"]:  # Skip latitude and longitude as they are already in required_view
            required_view.loc[index, col] = val  # Add each non-zero soil variable to required_view

required_view.to_csv("./required_view_soil_worldclim.csv", index=False)

Processing: -38.6, -72.4, 9/1/1996 to 8/30/2004
Processing: -37.40916667, -73.26694444, 7/6/1995 to 7/4/2003
Processing: -37.03333333, -72.23333333, 7/1/1997 to 6/29/2005
Processing: -35.87083333, -71.97527778, 8/1/1997 to 7/30/2005
Processing: -35.30694444, -72.39472222, 7/1/1993 to 6/29/2001
Processing: -34.04158333, 23.15783333, 8/24/2007 to 8/22/2015
Processing: -34.04070278, 23.17475278, 4/30/2009 to 4/28/2017
Processing: -34.03802778, 23.16933333, 12/2/2008 to 11/30/2016
Processing: -34.03073611, 24.1727, 10/9/2014 to 10/7/2022
Processing: -34.03038611, 23.18104444, 6/11/2013 to 6/9/2021
Processing: -34.02075, 23.0809, 6/25/2014 to 6/23/2022
Processing: -34.019475, 24.023675, 4/3/2013 to 4/1/2021
Processing: -34.017, 23.91247222, 8/28/2008 to 8/26/2016
Processing: -34.01561944, 23.07491389, 6/27/2012 to 6/25/2020
Processing: -34.01098333, 21.25663889, 9/4/2015 to 9/2/2023
Processing: -34.00194722, 21.27376111, 11/24/2011 to 11/22/2019
Processing: -34.00083611, 23.15783333, 11/20/

KeyboardInterrupt: 

In [39]:
required_view.iloc[138,:]

latitude                        17.8125
longitude                    -93.754167
date_range         [3/4/2008, 3/2/2016]
wm_prec1_autumn               95.353865
wm_prec1_spring              131.738856
wm_prec1_summer              165.535587
wm_prec1_winter               15.046804
wm_prec2_autumn               55.711474
wm_prec2_spring                62.29627
wm_prec2_summer               129.93185
wm_prec2_winter                9.176428
wm_tmax1_autumn               27.231105
wm_tmax1_spring               27.060723
wm_tmax1_summer               28.373607
wm_tmax1_winter               23.134061
wm_tmin1_autumn               10.072494
wm_tmin1_spring               10.300729
wm_tmin1_summer               15.791479
wm_tmin1_winter                5.773543
wm_tmax2_autumn               25.905827
wm_tmax2_spring                    26.0
wm_tmax2_summer               27.166666
wm_tmax2_winter               22.166666
wm_bio2                       13.328838
wm_bio5                       28.405827


In [31]:
required_view.to_csv("./required_view_soil_worldclim.csv", index=False)

In [ ]:
urls = [
    "https://maps.isric.org/mapserv?map=/map/ocd.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=ocd_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/ocs.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=ocs_0-30cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/bdod.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=bdod_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/clay.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=clay_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/cfvo.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=cfvo_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/sand.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=sand_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/silt.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=silt_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/wv0010.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=wv0010_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/wv0033.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=wv0033_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/wv1500.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=wv1500_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/cec.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=cec_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/nitrogen.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=nitrogen_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/soc.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=soc_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/phh2o.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=phh2o_0-5cm_mean&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
    "https://maps.isric.org/mapserv?map=/map/wrb.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage&COVERAGEID=MostProbable&FORMAT=image/tiff&SUBSET=long({lon_min},{lon_max})&SUBSET=lat({lat_min},{lat_max})&SUBSETTINGCRS=http://www.opengis.net/def/crs/EPSG/0/4326&OUTPUTCRS=http://www.opengis.net/def/crs/EPSG/0/4326",
]
# Function to download TIFF files for each URL
def download_tiff(url, lat_min, lat_max, lon_min, lon_max):
    # Format the URL with the coordinates
    formatted_url = url.format(lat_min=lat_min, lat_max=lat_max, lon_min=lon_min, lon_max=lon_max)

    # Send a GET request to the URL
    response = requests.get(formatted_url)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Extract the file name from the URL (e.g., using the coverage ID)
        coverage_id = formatted_url.split("COVERAGEID=")[1].split("&")[0]
        file_name = f"{coverage_id}.tif"

        # Save the content (image) to a file
        with open(file_name, "wb") as f:
            f.write(response.content)

        print(f"Downloaded: {file_name}")
    else:
        print(f"Failed to download {formatted_url}. Status code: {response.status_code}")

# Loop through each URL and download the corresponding TIFF file
for index, row in required_view.iloc[:5,:].iterrows():
    lat_center, lon_center = row["latitude"], row["longitude"]
    lat_min = lat_center - 0.05 / 2
    lat_max = lat_center + 0.05 / 2
    lon_min = lon_center - 0.05 / 2
    lon_max = lon_center + 0.05 / 2
    for url in urls:
        download_tiff(url, lat_min, lat_max, lon_min, lon_max)
    final_df = None

    # Iterate through the directory to read all the .tif files
    for root, dirs, files in os.walk("./soilgrids/"):
        for file in files:
            if file.endswith(".tif") and file!= "MostProbable.tif":
                # Extract the variable name from the file name
                name_var = file.split("_")[0]

                # Open the raster file using rioxarray
                raster = rioxarray.open_rasterio(
                    f"./soilgrids/{file}",
                    mask_and_scale=True,
                )

                # Squeeze to remove singleton dimensions and drop unnecessary variables
                rds = raster.squeeze().drop_vars("spatial_ref").drop_vars("band")
                rds.name = name_var

                # Convert the raster to a DataFrame
                df = rds.to_dataframe().reset_index()

                # Filter out rows with NaN values in the raster data column
                df_filtered = df[df[rds.name].notna()]

                # Create a 'location' column using latitude and longitude
                df_filtered['location'] = df_filtered.apply(
                    lambda row: f"lat{row['x']:.5f}_lon{row['y']:.5f}", axis=1
                )

                # Drop the 'x' and 'y' columns as they are not needed anymore
                df_filtered.drop(columns=["x", "y"], inplace=True, axis=1)

                # Select only the 'location' and the current variable column
                df = df_filtered[['location', rds.name]]

    #             # Merge the current df with the final dataframe
                if final_df is None:
                    final_df = df
                else:
                    final_df = final_df.merge(df, on='location', how='outer')
    final_df

In [40]:
big.head()

,env,TestId,Specie,latitude,longitude,Productivity (y),data_final,date_final,age (years),Feature,...,bio_2,bio_3,bio_4,bio_5,bio_6,bio_7,bio_8,bio_9,elev,date_range
0,1,204118A,PGRN,-38.6,-72.4,7.12,9/1/1996,8/30/2004,8,NaN,...,12.225,55.821918,363.56897,24.9,3.0,21.9,7.9,16.0,181,"9/1/1996, 8/30/2004"
1,1,204118A,PGRN,-38.6,-72.4,7.20,9/1/1996,8/30/2004,8,NaN,...,12.225,55.821918,363.56897,24.9,3.0,21.9,7.9,16.0,181,"9/1/1996, 8/30/2004"
2,1,204118A,PGRN,-38.6,-72.4,7.33,9/1/1996,8/30/2004,8,NaN,...,12.225,55.821918,363.56897,24.9,3.0,21.9,7.9,16.0,181,"9/1/1996, 8/30/2004"
3,1,204118A,PGRN,-38.6,-72.4,7.75,9/1/1996,8/30/2004,8,NaN,...,12.225,55.821918,363.56897,24.9,3.0,21.9,7.9,16.0,181,"9/1/1996, 8/30/2004"
4,1,204118A,PGRN,-38.6,-72.4,7.82,9/1/1996,8/30/2004,8,NaN,...,12.225,55.821918,363.56897,24.9,3.0,21.9,7.9,16.0,181,"9/1/1996, 8/30/2004"


In [44]:
required_view.head()

,latitude,longitude,date_range,wm_prec1_autumn,wm_prec1_spring,wm_prec1_summer,wm_prec1_winter,wm_prec2_autumn,wm_prec2_spring,wm_prec2_summer,...,nitrogen,ocd,ocs,phh2o,sand,silt,soc,wv0010,wv0033,wv1500
0,17.8125,-93.754167,"[9/1/1996, 8/30/2004]",159.771858,153.966496,60.852583,326.154303,79.575621,81.201653,29.641108,...,585.0,360.0,61.0,54.0,423.0,285.0,408.0,373.0,320.0,164.0
1,17.8125,-93.754167,"[7/6/1995, 7/4/2003]",191.079978,150.033668,47.091699,415.585844,96.704505,63.717432,20.878716,...,585.0,360.0,61.0,54.0,423.0,285.0,408.0,373.0,320.0,164.0
2,17.8125,-93.754167,"[7/1/1997, 6/29/2005]",187.691517,128.757154,46.221770,337.353505,49.177303,57.697603,18.391200,...,585.0,360.0,61.0,54.0,423.0,285.0,408.0,373.0,320.0,164.0
3,17.8125,-93.754167,"[8/1/1997, 7/30/2005]",135.033440,106.014488,20.312300,251.468277,30.449775,42.322747,6.222922,...,585.0,360.0,61.0,54.0,423.0,285.0,408.0,373.0,320.0,164.0
4,17.8125,-93.754167,"[7/1/1993, 6/29/2001]",81.987113,85.244236,14.242917,217.227977,39.331023,25.246566,6.489844,...,585.0,360.0,61.0,54.0,423.0,285.0,408.0,373.0,320.0,164.0


In [46]:
big.columns = big.columns.str.strip()
required_view.columns = required_view.columns.str.strip()
big["latitude"] = big["latitude"].astype(str).str.strip()
required_view["latitude"] = required_view["latitude"].astype(str).str.strip()

In [48]:
print(big.dtypes)

env                   int64
TestId               object
Specie               object
latitude             object
longitude           float64
Productivity (y)    float64
data_final           object
date_final           object
age (years)           int64
Feature             float64
bio_1               float64
bio_10              float64
bio_11              float64
bio_12                int64
bio_13                int64
bio_14                int64
bio_15              float64
bio_16                int64
bio_17                int64
bio_18                int64
bio_19                int64
bio_2               float64
bio_3               float64
bio_4               float64
bio_5               float64
bio_6               float64
bio_7               float64
bio_8               float64
bio_9               float64
elev                  int64
date_range           object
dtype: object


In [66]:
required_view[["latitude","longitude","data_final","date_final"]].value_counts() 

latitude  longitude   data_final  date_final
17.8125   -93.754167  1992-02-01  2000-01-30    6
                      1990-03-01  1998-02-27    4
                      1988-04-01  1996-03-30    3
                      1988-10-01  1996-09-29    3
                      1985-11-01  1993-10-30    3
                                               ..
                      2015-10-23  2023-10-21    1
                      2015-12-10  2023-12-08    1
                      2016-02-09  2024-02-07    1
                      2016-02-23  2024-02-21    1
                      2016-08-02  2024-07-31    1
Name: count, Length: 240, dtype: int64

In [72]:
big[["latitude", "longitude", "data_final", "date_final"]].drop_duplicates().value_counts()

latitude    longitude   data_final  date_final
 17.814889  -93.754444  2003-07-11  2011-07-09    1
-38.600000  -72.400000  1996-09-01  2004-08-30    1
-37.409167  -73.266944  1995-07-06  2003-07-04    1
-37.033333  -72.233333  1997-07-01  2005-06-29    1
-35.870833  -71.975278  1997-08-01  2005-07-30    1
                                                 ..
-34.020750   23.080900  2014-06-25  2022-06-23    1
-34.030386   23.181044  2013-06-11  2021-06-09    1
-34.030736   24.172700  2014-10-09  2022-10-07    1
-34.038028   23.169333  2008-12-02  2016-11-30    1
-34.040703   23.174753  2009-04-30  2017-04-28    1
Name: count, Length: 310, dtype: int64

In [73]:
required_view[["latitude","longitude","data_final","date_final"]].drop_duplicates().value_counts()

latitude  longitude   data_final  date_final
17.8125   -93.754167  1981-11-01  1989-10-30    1
                      1982-02-01  1990-01-30    1
                      1982-05-01  1990-04-29    1
                      1982-07-01  1990-06-29    1
                      1982-08-01  1990-07-30    1
                                               ..
                      2015-10-23  2023-10-21    1
                      2015-12-10  2023-12-08    1
                      2016-02-09  2024-02-07    1
                      2016-02-23  2024-02-21    1
                      2016-08-02  2024-07-31    1
Name: count, Length: 240, dtype: int64

In [69]:
# Find duplicates based on the relevant columns (latitude, longitude, data_final, date_final)
duplicates_by_keys = required_view[required_view.duplicated(subset=["latitude", "longitude", "data_final", "date_final"], keep=False)]
print(duplicates_by_keys)

     latitude  longitude                date_range  wm_prec1_autumn  \
16    17.8125 -93.754167  [11/20/2007, 11/18/2015]        68.710003   
27    17.8125 -93.754167    [10/1/1986, 9/29/1994]       112.467858   
31    17.8125 -93.754167     [2/1/1992, 1/30/2000]        93.316779   
34    17.8125 -93.754167     [3/1/1988, 2/28/1996]       110.798613   
36    17.8125 -93.754167     [2/1/1996, 1/30/2004]       107.852804   
..        ...        ...                       ...              ...   
298   17.8125 -93.754167   [11/1/1984, 10/30/1992]        86.774701   
301   17.8125 -93.754167     [6/1/1997, 5/30/2005]       199.129057   
303   17.8125 -93.754167     [5/1/1983, 4/29/1991]       168.490958   
305   17.8125 -93.754167     [6/1/1994, 5/30/2002]       160.155732   
308   17.8125 -93.754167    [10/1/1985, 9/29/1993]       121.555087   

     wm_prec1_spring  wm_prec1_summer  wm_prec1_winter  wm_prec2_autumn  \
16        102.156300        68.906463        91.639772        34.047443 

In [ ]:
# Ensure latitude & longitude are floats
big["latitude"] = big["latitude"].astype(float)
required_view["latitude"] = required_view["latitude"].astype(float)

big["longitude"] = big["longitude"].astype(float)
required_view["longitude"] = required_view["longitude"].astype(float)

# Ensure date columns are in datetime format
big["data_final"] = pd.to_datetime(big["data_final"], errors="coerce")
required_view["data_final"] = pd.to_datetime(required_view["data_final"], errors="coerce")

big["date_final"] = pd.to_datetime(big["date_final"], errors="coerce")
required_view["date_final"] = pd.to_datetime(required_view["date_final"], errors="coerce")

In [106]:
pd.set_option('display.max_columns', None)

In [116]:
required_view.head()

,latitude,longitude,date_range,wm_prec1_autumn,wm_prec1_spring,wm_prec1_summer,wm_prec1_winter,wm_prec2_autumn,wm_prec2_spring,wm_prec2_summer,wm_prec2_winter,wm_tmax1_autumn,wm_tmax1_spring,wm_tmax1_summer,wm_tmax1_winter,wm_tmin1_autumn,wm_tmin1_spring,wm_tmin1_summer,wm_tmin1_winter,wm_tmax2_autumn,wm_tmax2_spring,wm_tmax2_summer,wm_tmax2_winter,wm_bio2,wm_bio5,wm_bio6,wm_bio7,wm_bio3,wm_bio12,wm_bio13,wm_bio14,wm_bio16,wm_bio17,wm_bio18,wm_bio19,data_final,date_final
0,-38.600000,-72.400000,"[9/1/1996, 8/30/2004]",159.771858,153.966496,60.852583,326.154303,79.575621,81.201653,29.641108,214.564616,21.732737,19.694051,25.172490,12.715474,4.138686,3.607737,8.000000,2.760109,20.000000,19.000000,21.552408,12.000000,12.017911,25.172490,2.663487,22.509002,0.530812,1208.573470,301.741827,13.282245,509.672407,100.791624,439.426175,441.487279,9/1/1996,8/30/2004
1,-37.409167,-73.266944,"[7/6/1995, 7/4/2003]",191.079978,150.033668,47.091699,415.585844,96.704505,63.717432,20.878716,259.001336,20.055129,18.490851,22.751127,13.103730,6.058771,4.585023,9.000673,3.792981,19.021679,17.927505,22.021679,11.470862,10.098168,22.751127,3.737426,19.013701,0.515069,1362.673521,415.585844,4.867389,680.982536,94.983232,527.327670,549.069543,7/6/1995,7/4/2003
2,-37.033333,-72.233333,"[7/1/1997, 6/29/2005]",187.691517,128.757154,46.221770,337.353505,49.177303,57.697603,18.391200,215.218964,23.125000,21.750000,27.613760,13.425926,5.321928,4.717762,10.444444,3.694891,22.000000,21.000000,25.000000,11.833333,12.029924,27.613760,3.514112,24.099649,0.487999,1097.062553,339.229738,4.736680,553.038430,91.378334,516.042599,516.042599,7/1/1997,6/29/2005
3,-35.870833,-71.975278,"[8/1/1997, 7/30/2005]",135.033440,106.014488,20.312300,251.468277,30.449775,42.322747,6.222922,122.456269,24.670309,23.171241,29.223051,14.386480,6.168444,5.731876,11.372779,4.206112,23.072495,22.362471,26.072495,13.166667,12.261349,29.223051,4.095001,25.128050,0.479743,751.755998,261.001174,2.545439,404.053425,52.220430,308.388293,308.388293,8/1/1997,7/30/2005
4,-35.306944,-72.394722,"[7/1/1993, 6/29/2001]",81.987113,85.244236,14.242917,217.227977,39.331023,25.246566,6.489844,111.168608,21.524051,20.569555,24.708968,14.238954,8.021453,6.844664,10.693632,4.953509,20.054386,19.878656,23.054386,13.271935,9.901755,24.708968,4.953509,19.755459,0.490689,603.113818,228.247552,2.537238,352.698599,19.991884,286.853651,286.853651,7/1/1993,6/29/2001
